# Lab (INTENDED)

Instructor-complete version of the student lab. The differential-expression table is **precomputed** and loaded here. Do not recompute the t-tests from the walkthrough. Students use the [lab prompts](Module11_lab_prompts.md) page for the point of each step.

## Learning objectives

- Draw a volcano plot from a saved DE table
- Run Enrichr on the significant gene list
- Plot expression for the 10 genes with largest |log2FC| in one Hallmark set

## Biological framing

You start from a finished differential-expression table for the same acute-HIV PBMC study (**GSE199911**): 29 infected versus 46 uninfected samples, tested gene-by-gene on log2 TPM. Your job is to **see** the genome-wide pattern (volcano) and then ask whether the significant genes concentrate in known Hallmark programs. Finally, check that a chosen Enrichr term’s strongest fold-change genes actually separate by infection status across people—not just as a pathway name on a list.


In [ ]:
# Setting up the Colab environment. DO NOT EDIT!
try:
  from applied_biostats import setup_environment
except ImportError:
  !pip -q install applied-biostats-helper
  from applied_biostats import setup_environment
finally:
  grader = setup_environment('Module11_lab')

!pip -q install gseapy==1.3.2


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import gseapy


## Load the saved analysis tables

These cells stay filled in for students. Nothing below rebuilds the gene-wise t-tests.


In [ ]:
samples = pd.read_csv('gse199911_samples.csv')
log2 = pd.read_csv('gse199911_log2tpm.csv.gz').set_index('gene_symbol')
results = pd.read_csv('gse199911_de_results.csv')

print('Unit of analysis: sample')
print('n samples:', len(samples))
print(samples['group'].value_counts())
print('results genes:', len(results), 'fdr<0.05:', int((results['fdr'] < 0.05).sum()))
results.head()


## Q1. Volcano plot

Scatter log2 fold change (x) against `-log10(FDR)` (y). Highlight genes with FDR < 0.05.


In [ ]:
plot_df = results.copy()
plot_df['neglog10_fdr'] = -np.log10(plot_df['fdr'].clip(lower=1e-300))
plot_df['significant'] = plot_df['fdr'] < 0.05

plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=plot_df,
    x='log2fc',
    y='neglog10_fdr',
    hue='significant',
    palette={False: 'lightgray', True: 'crimson'},
    linewidth=0,
    s=12,
    alpha=0.7,
)
plt.axhline(-np.log10(0.05), ls='--', color='black', lw=1)
plt.xlabel('log2 fold change (HIV - control)')
plt.ylabel('-log10 FDR')
plt.title('Volcano plot: acute HIV vs control')
plt.tight_layout()
plt.show()


## Q2A. Enrichr on the significant list

Build the significant gene list (FDR < 0.05 and |log2FC| > 1). Run Enrichr against `MSigDB_Hallmark_2020` with `outdir=None`. Inspect the top terms.


In [ ]:
sig = results.loc[(results['fdr'] < 0.05) & (results['log2fc'].abs() > 1), 'gene_symbol'].tolist()
print('genes sent to Enrichr:', len(sig))

enr = gseapy.enrichr(
    gene_list=sig,
    gene_sets='MSigDB_Hallmark_2020',
    organism='Human',
    outdir=None,
)
enrich_df = enr.results.sort_values('Adjusted P-value')
print(enrich_df.head(5)[['Term', 'Adjusted P-value', 'Overlap']])
enrich_df.head(10)


## Q2B. Plot the top |log2FC| genes from one Hallmark term

Pick one significant term from `enrich_df`. Among overlap genes in `results`, take the 10 with largest absolute log2FC. Plot their log2 TPM across samples, colored by group.


In [ ]:
term = enrich_df.iloc[0]['Term']
term_genes = set(enrich_df.iloc[0]['Genes'].split(';'))
print('chosen term:', term)
print('overlap genes:', len(term_genes))

top10 = (
    results.loc[results['gene_symbol'].isin(term_genes)]
    .assign(abs_fc=lambda d: d['log2fc'].abs())
    .sort_values('abs_fc', ascending=False)
    .head(10)
)
print(top10[['gene_symbol', 'log2fc', 'fdr']])

long = (
    log2.loc[top10['gene_symbol']]
    .reset_index()
    .melt(id_vars='gene_symbol', var_name='sample_id', value_name='log2tpm')
    .merge(samples[['sample_id', 'group']], on='sample_id', how='left')
)

plt.figure(figsize=(10, 5))
sns.stripplot(data=long, x='gene_symbol', y='log2tpm', hue='group', dodge=True, alpha=0.7)
sns.boxplot(
    data=long, x='gene_symbol', y='log2tpm', hue='group', dodge=True,
    fliersize=0, boxprops=dict(alpha=0.3),
)
plt.xticks(rotation=45, ha='right')
plt.ylabel('log2(TPM + 1)')
plt.title(f'Top |log2FC| genes in {term}')
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[:2], labels[:2], title='group')
plt.tight_layout()
plt.show()
